# Criação do Dataset com Strings Originais (para comparação ML raiz vs LLMs)

**Objetivo**: Construir `train_strings.parquet` e `test_strings.parquet` a partir dos artefatos
já existentes do pipeline, restaurando as colunas de texto originais (Origem, Destino,
Grupo de Mercadoria, Grupo Mercadoria Conteinerizada) que foram substituídas por embeddings.

**Estratégia**:
1. Carrega o dataset combinado (`train.parquet` + `test.parquet`) — já tem todos os features engineered
2. Busca os valores string dessas 4 colunas nos parquets nível-carga (`data/processed/df_{ano}.parquet`)
3. Agrega strings por atracação (carga mais pesada = VLPesoCargaBruta)
4. Faz merge de volta no dataset combinado
5. Novo split: **200 amostras aleatórias** para teste, o restante para treino

As colunas de embedding originais são **mantidas** (Eduardo: "se já tiver redundante, mantenha").


In [1]:
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path

# ── Configurações ──────────────────────────────────────────────────────────────
ANOS        = [2018, 2019, 2020, 2021, 2022, 2023, 2024]
RANDOM_SEED = 42
N_TEST      = 200

BASE_DIR    = Path("antaq-turnaround-data-v1.2/data")
OUTPUT_DIR  = Path("data_input")   # na raiz do projeto
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ID_ATRAC  = "IDAtracacao"
PESO_COL  = "VLPesoCargaBruta"

TARGET_COLS = [
    "TEstadia", "TEsperaAtracacao", "TAtracado",
    "TEsperaInicioOp", "TOperacao", "TEsperaDesatracacao",
]
TEXT_COLS = [
    "Origem",
    "Destino",
    "Grupo de Mercadoria",
    "Grupo Mercadoria Conteinerizada",
]

print("Configuração OK")
print(f"Output: {OUTPUT_DIR.resolve()}")


Configuração OK
Output: /home/jonnathan/Área de trabalho/Projetos/3_/data_input


## 1. Carregar train e test originais

`train.parquet` e `test.parquet` são carregados separadamente — **sem concatenação**.
O test set é amostrado em 200 linhas aleatórias do `test.parquet` original.
Ambos já têm todos os features do pipeline (OHE+SUM, numéricos, categóricos, embeddings de rota e mercadoria, imputação aplicada).

In [2]:
print("Carregando train.parquet...")
df_train = pq.read_table(str(BASE_DIR / "output/train.parquet")).to_pandas()
print(f"  train: {df_train.shape}")

print("Carregando test.parquet...")
df_test = pq.read_table(str(BASE_DIR / "output/test.parquet")).to_pandas()
print(f"  test : {df_test.shape}")

# Amostra 200 do test set original
df_test = df_test.sample(n=N_TEST, random_state=RANDOM_SEED).reset_index(drop=True)
print(f"\nTest set amostrado: {df_test.shape}")

# Confirmar que TEXT_COLS estão AUSENTES (foram substituídas por embeddings)
print("\nVerificando ausência das colunas texto originais:")
for col in TEXT_COLS:
    status = "PRESENTE" if col in df_train.columns else "AUSENTE (correto)"
    print(f"  {col}: {status}")


Carregando train.parquet...
  train: (391460, 1613)
Carregando test.parquet...
  test : (97866, 1613)

Test set amostrado: (200, 1613)

Verificando ausência das colunas texto originais:
  Origem: AUSENTE (correto)
  Destino: AUSENTE (correto)
  Grupo de Mercadoria: AUSENTE (correto)
  Grupo Mercadoria Conteinerizada: AUSENTE (correto)


## 2. Buscar strings originais nos parquets nível-carga

Os parquets `data/processed/df_{ano}.parquet` são nível de carga (1 linha por IDCarga)
e já têm os textos enriquecidos gerados pelo `load.py`:
- `Origem` / `Destino`: texto enriquecido ("Nome, País, Continente, BlocoEconômico")
- `Grupo de Mercadoria`: do join com `Mercadoria.txt`
- `Grupo Mercadoria Conteinerizada`: do join com `MercadoriaConteinerizada.txt`

Carregamos apenas as 6 colunas necessárias para economia de RAM.

In [3]:
COLS_NEEDED = [ID_ATRAC, PESO_COL] + TEXT_COLS

frames = []
for ano in ANOS:
    p = BASE_DIR / "processed" / f"df_{ano}.parquet"
    if not p.exists():
        print(f"  {ano}: NÃO ENCONTRADO — pulando")
        continue

    # Lê apenas as colunas que existem no parquet deste ano
    schema = pq.read_schema(str(p))
    avail  = [c for c in COLS_NEEDED if c in schema.names]
    df_ano = pq.read_table(str(p), columns=avail).to_pandas()
    frames.append(df_ano)
    print(f"  {ano}: {len(df_ano):,} linhas | colunas lidas: {avail}")
    del df_ano

df_cargo = pd.concat(frames, ignore_index=True)
del frames

print(f"\nTotal linhas de carga carregadas: {len(df_cargo):,}")
print(f"Atracações únicas: {df_cargo[ID_ATRAC].nunique():,}")


  2018: 2,079,460 linhas | colunas lidas: ['IDAtracacao', 'VLPesoCargaBruta', 'Origem', 'Destino', 'Grupo de Mercadoria', 'Grupo Mercadoria Conteinerizada']
  2019: 2,013,749 linhas | colunas lidas: ['IDAtracacao', 'VLPesoCargaBruta', 'Origem', 'Destino', 'Grupo de Mercadoria', 'Grupo Mercadoria Conteinerizada']
  2020: 2,103,791 linhas | colunas lidas: ['IDAtracacao', 'VLPesoCargaBruta', 'Origem', 'Destino', 'Grupo de Mercadoria', 'Grupo Mercadoria Conteinerizada']
  2021: 2,338,540 linhas | colunas lidas: ['IDAtracacao', 'VLPesoCargaBruta', 'Origem', 'Destino', 'Grupo de Mercadoria', 'Grupo Mercadoria Conteinerizada']
  2022: 2,262,416 linhas | colunas lidas: ['IDAtracacao', 'VLPesoCargaBruta', 'Origem', 'Destino', 'Grupo de Mercadoria', 'Grupo Mercadoria Conteinerizada']
  2023: 2,122,644 linhas | colunas lidas: ['IDAtracacao', 'VLPesoCargaBruta', 'Origem', 'Destino', 'Grupo de Mercadoria', 'Grupo Mercadoria Conteinerizada']
  2024: 2,317,176 linhas | colunas lidas: ['IDAtracacao', 

## 3. Agregar strings para nível de atracação

Estratégia: para cada `IDAtracacao`, pegar o valor de texto da **carga mais pesada**
(`VLPesoCargaBruta` máximo). Esta é a mesma lógica usada no notebook original e é
consistente com a média ponderada por massa dos embeddings.

Para `Grupo Mercadoria Conteinerizada`: muitas cargas têm valor nulo/inaplicável.
Usamos a carga mais pesada que tenha um valor real; se nenhuma tiver, fica "Não se aplica".

In [4]:
NA_VALUE = "Não se aplica"

def aggregate_text_col(df_cargo, id_col, peso_col, text_col, na_value=NA_VALUE):
    """
    Para cada id_col, retorna o valor de text_col da linha com maior peso.
    Prioriza linhas com valor real (não NA) antes de considerar o peso.
    """
    df = df_cargo[[id_col, peso_col, text_col]].copy()
    df[text_col] = df[text_col].fillna(na_value).astype(str).str.strip()

    # Flag: valor real (não NA, não vazio, não 'nan')
    bad = {na_value.lower(), "", "nan", "none"}
    df["_valid"] = ~df[text_col].str.lower().isin(bad)

    # Ordena: primeiro válidos (desc), depois por peso (desc)
    df = df.sort_values(
        ["_valid", peso_col],
        ascending=[False, False],
        na_position="last",
    )
    result = (
        df.drop_duplicates(subset=[id_col], keep="first")
          [[id_col, text_col]]
    )
    return result

# Agrega cada coluna texto separadamente para poder aplicar a lógica de desdiluição
dfs = []
for col in TEXT_COLS:
    if col not in df_cargo.columns:
        print(f"  {col}: coluna ausente nos parquets — será preenchida com '{NA_VALUE}'")
        continue
    agg = aggregate_text_col(df_cargo, ID_ATRAC, PESO_COL, col)
    dfs.append(agg)
    n_valid = (agg[col] != NA_VALUE).sum()
    pct = n_valid / len(agg) * 100
    exemplo = agg[col][agg[col] != NA_VALUE].iloc[0] if n_valid > 0 else NA_VALUE
    print(f"  {col}: {n_valid:,} valores reais ({pct:.1f}%) | ex: '{exemplo[:60]}'")

# Junta todas as colunas em um único DataFrame de strings por atracação
df_strings = dfs[0]
for agg in dfs[1:]:
    df_strings = df_strings.merge(agg, on=ID_ATRAC, how="outer")

# Garante que atracações sem carga válida ficam com NA_VALUE
for col in TEXT_COLS:
    if col in df_strings.columns:
        df_strings[col] = df_strings[col].fillna(NA_VALUE)

print(f"\ndf_strings: {df_strings.shape} ({df_strings[ID_ATRAC].nunique():,} atracações únicas)")


  Origem: 489,695 valores reais (100.0%) | ex: 'BRASIL, AMÉRICA DO SUL, Mercosul'
  Destino: 489,695 valores reais (100.0%) | ex: 'BRASIL, AMÉRICA DO SUL, Mercosul'
  Grupo de Mercadoria: 489,695 valores reais (100.0%) | ex: 'Transações especiais'
  Grupo Mercadoria Conteinerizada: 63,219 valores reais (12.9%) | ex: 'Plásticos e suas obras'

df_strings: (489695, 5) (489,695 atracações únicas)


## 4. Merge das strings no dataset combinado

In [5]:
def merge_strings(df, df_strings, id_col, text_cols, na_value=NA_VALUE):
    n_antes = len(df)
    df = df.merge(df_strings, on=id_col, how="left")
    assert len(df) == n_antes
    for col in text_cols:
        if col in df.columns:
            n_na = df[col].isna().sum()
            if n_na:
                print(f"  AVISO: {col} tem {n_na:,} NaN — preenchendo com '{na_value}'")
            df[col] = df[col].fillna(na_value)
    return df

df_train = merge_strings(df_train, df_strings, ID_ATRAC, TEXT_COLS)
df_test  = merge_strings(df_test,  df_strings, ID_ATRAC, TEXT_COLS)

print(f"Train: {df_train.shape}")
print(f"Test : {df_test.shape}")


Train: (391460, 1617)
Test : (200, 1617)


## 4.5. Remover colunas de embedding

O objetivo deste dataset é comparar ML raiz vs LLMs usando features tabulares + strings.
Os embeddings pré-computados são removidos — o modelo de LLM gerará suas próprias representações
a partir das strings; o ML raiz usará apenas as features tabulares (OHE+SUM, numéricas, categóricas).

In [6]:
# Remove embeddings e targets não preditos de ambos
PREDICTED_TARGETS = ['TOperacao', 'TAtracado']
OTHER_TARGETS = [c for c in TARGET_COLS if c not in PREDICTED_TARGETS]
emb_cols = [c for c in df_train.columns if "_embedding_" in c]

drop_cols = emb_cols + OTHER_TARGETS

df_train = df_train.drop(columns=[c for c in drop_cols if c in df_train.columns])
df_test  = df_test.drop(columns=[c for c in drop_cols if c in df_test.columns])

print(f"Embeddings removidos        : {len(emb_cols)}")
print(f"Targets não preditos removidos: {OTHER_TARGETS}")
print(f"Targets mantidos            : {PREDICTED_TARGETS}")
print(f"Train: {df_train.shape}")
print(f"Test : {df_test.shape}")

Embeddings removidos        : 1536
Targets não preditos removidos: ['TEstadia', 'TEsperaAtracacao', 'TEsperaInicioOp', 'TEsperaDesatracacao']
Targets mantidos            : ['TOperacao', 'TAtracado']
Train: (391460, 77)
Test : (200, 77)


## 5. Criar test set (200 amostras aleatórias) e train set

Instrução Eduardo: test set com **200 casos amostrados aleatoriamente** e fora do treino.
O restante (~489k atracações) compõe o train set.

In [7]:
# Sanity check: sem sobreposição entre train e test
overlap = set(df_train[ID_ATRAC]) & set(df_test[ID_ATRAC])
assert len(overlap) == 0, f"ERRO: {len(overlap)} atracações em treino E teste!"

print(f"Train set: {len(df_train):,} atracações × {df_train.shape[1]} colunas")
print(f"Test set : {len(df_test):,} atracações × {df_test.shape[1]} colunas")
print(f"\nDistribuição dos targets:")
for col in PREDICTED_TARGETS:
    tr_med = df_train[col].median()
    te_med = df_test[col].median()
    print(f"  {col:25s}  train median={tr_med:7.2f}h  |  test median={te_med:7.2f}h")


Train set: 391,460 atracações × 77 colunas
Test set : 200 atracações × 77 colunas

Distribuição dos targets:
  TOperacao                  train median=   8.92h  |  test median=   8.84h
  TAtracado                  train median=  14.90h  |  test median=  16.87h


## 6. EDA rápida — amostra do test set (colunas texto + targets)

In [ ]:
cols_mostrar = TEXT_COLS + PREDICTED_TARGETS
df_test[cols_mostrar].sample(5, random_state=RANDOM_SEED)
[cols_mostrar].sample(5, random_state=RANDOM_SEED)

,Origem,Destino,Grupo de Mercadoria,Grupo Mercadoria Conteinerizada,TOperacao,TAtracado
95,"Rio Grande, BRASIL, AMÉRICA DO SUL, Mercosul","Wilmington, ESTADOS UNIDOS, AMÉRICA DO NORTE",Pastas de madeira ou de outras matérias fibros...,Não se aplica,51.283333,54.166668
15,"Belém, BRASIL, AMÉRICA DO SUL, Mercosul","Vila do Conde, BRASIL, AMÉRICA DO SUL, Mercosul","Combustíveis minerais, óleos minerais e produt...",Não se aplica,8.133333,18.400000
30,"Terminais Fluviais do Brasil, BRASIL, AMÉRICA ...","ABI Miritituba, BRASIL, AMÉRICA DO SUL, Mercosul","Combustíveis minerais, óleos minerais e produt...",Não se aplica,160.916672,165.500000
158,"Terbian - Terminal Bianchini, BRASIL, AMÉRICA ...","Não Identificado, CORÉIA DO SUL, ÁSIA",Resíduos e desperdícios das indústrias aliment...,Não se aplica,45.916668,62.583332
128,"ETC Miritituba, BRASIL, AMÉRICA DO SUL, Mercosul","Terminal Portuário Graneleiro de Barcarena, BR...",Cereais,Não se aplica,1.733333,1.966667


In [10]:
TARGET_COLS = [
    "TEstadia", "TEsperaAtracacao", "TAtracado",
    "TEsperaInicioOp", "TOperacao", "TEsperaDesatracacao",
]

In [11]:
df_train.columns

Index(['IDAtracacao', 'Porto Atracação', 'Complexo Portuário',
       'Tipo da Autoridade Portuária', 'Tipo de Operação',
       'Tipo de Navegação da Atracação', 'Nacionalidade do Armador',
       'Município', 'UF', 'SGUF', 'Região Geográfica', 'Região Hidrográfica',
       'Instalação Portuária em Rio', 'Natureza da Carga',
       'Percurso Transporte Interiores', 'STNaturezaCarga',
       'Carga Geral Acondicionamento', 'lon', 'lat', 'mes_sin', 'mes_cos',
       'Ano', 'Mes_num', 'DiaSemana', 'TOperacao', 'TAtracado',
       'VLPesoCargaBruta', 'TEU', 'QTCarga', 'VLPesoCargaConteinerizada_total',
       'valor_mov_regiao_total', 'valor_mov_rio_total',
       'valor_mov_hidrovia_total', 'valor_mov_total_hidrografia',
       'ohe_Tipo_Operação_da_Carga__Abastecimento',
       'ohe_Tipo_Operação_da_Carga__Apoio',
       'ohe_Tipo_Operação_da_Carga__Baldeação_de_Carga_Estrangeira_de_Passagem',
       'ohe_Tipo_Operação_da_Carga__Baldeação_de_Carga_Nacional',
       'ohe_Tipo_Operação_da

## 7. Salvar

In [12]:
train_path = OUTPUT_DIR / "train_strings.parquet"
test_path  = OUTPUT_DIR / "test_strings.parquet"

pq.write_table(pa.Table.from_pandas(df_train, preserve_index=False), str(train_path), compression="zstd")
pq.write_table(pa.Table.from_pandas(df_test,  preserve_index=False), str(test_path),  compression="zstd")

print("Salvo com sucesso!")
print(f"  {train_path.name}: {len(df_train):,} linhas × {df_train.shape[1]} colunas ({train_path.stat().st_size / 1e6:.0f} MB)")
print(f"  {test_path.name} : {len(df_test):,} linhas × {df_test.shape[1]} colunas ({test_path.stat().st_size / 1e6:.1f} MB)")
print(f"\nTargets mantidos             : {PREDICTED_TARGETS}")
print(f"Targets removidos            : {OTHER_TARGETS}")
print(f"Embeddings removidos         : {len(emb_cols)}")

Salvo com sucesso!
  train_strings.parquet: 391,460 linhas × 77 colunas (29 MB)
  test_strings.parquet : 200 linhas × 77 colunas (0.1 MB)

Targets mantidos             : ['TOperacao', 'TAtracado']
Targets removidos            : ['TEstadia', 'TEsperaAtracacao', 'TEsperaInicioOp', 'TEsperaDesatracacao']
Embeddings removidos         : 1536
